In [1]:
import pandas as pd

In [6]:
print(person.columns.tolist())

['SERIAL', 'PERNUM', 'SEX', 'AGE', 'RACE', 'RACED', 'PERWT', 'SEX_LABEL', 'RACE_LABEL']


In [3]:
employment = pd.read_csv("../data/processed/employment.csv")

print(employment.columns.tolist())

['SERIAL', 'PERNUM', 'EMPSTAT', 'EMPSTATD', 'OCC', 'IND', 'INCWAGE', 'EMPSTAT_LABEL']


In [5]:
person = pd.read_csv("../data/processed/person.csv")

person_eda = person[
    [
        "SERIAL",
        "PERNUM",
        "AGE",
        "SEX_LABEL"
    ]
].copy()

person_eda.to_csv(
    "../data/processed/person_eda.csv",
    index=False
)

print(person_eda.shape)

(3238311, 4)


In [7]:
employment = pd.read_csv("../data/processed/employment.csv")

employment_eda = employment[
    [
        "SERIAL",
        "PERNUM",
        "EMPSTAT_LABEL",
        "INCWAGE"
    ]
].copy()

employment_eda.to_csv(
    "../data/processed/employment_eda.csv",
    index=False
)

print(employment_eda.shape)

(3238311, 4)


In [9]:
import os

for f in [
    "../data/processed/person_eda.csv",
    "../data/processed/employment_eda.csv"
]:
    print(
        f,
        round(os.path.getsize(f)/1024/1024, 2),
        "MB"
    )

../data/processed/person_eda.csv 56.28 MB
../data/processed/employment_eda.csv 79.48 MB


In [8]:
import pandas as pd

person = pd.read_csv("../data/processed/person_eda.csv")
employment = pd.read_csv("../data/processed/employment_eda.csv")

df = pd.merge(person, employment, on=["SERIAL", "PERNUM"], how="inner")

df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df["INCWAGE"] = pd.to_numeric(df["INCWAGE"], errors="coerce")

df = df[
    (df["AGE"].notna()) &
    (df["AGE"] >= 18) &
    (df["AGE"] <= 80) &
    (df["SEX_LABEL"].notna()) &
    (df["EMPSTAT_LABEL"].notna())
].copy()

df["AGE_GROUP"] = pd.cut(
    df["AGE"],
    bins=[17, 24, 34, 44, 54, 64, 80],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
)

df["EMPLOYED_FLAG"] = (df["EMPSTAT_LABEL"] == "Employed").astype(int)

employment_rate_age_gender = (
    df.groupby(["AGE_GROUP", "SEX_LABEL"], observed=True)["EMPLOYED_FLAG"]
    .mean()
    .reset_index()
)

employment_rate_age_gender["Employment_Rate"] = employment_rate_age_gender["EMPLOYED_FLAG"] * 100

employment_rate_age_gender.to_csv(
    "../data/processed/employment_rate_age_gender_yifan.csv",
    index=False
)

income_gender_sample = df[
    (df["INCWAGE"] > 0) &
    (df["INCWAGE"] <= 300000)
][["SEX_LABEL", "INCWAGE"]].sample(
    n=50000,
    random_state=42
)

income_gender_sample.to_csv(
    "../data/processed/income_gender_sample_yifan.csv",
    index=False
)

print(employment_rate_age_gender.shape)
print(income_gender_sample.shape)

(12, 4)
(50000, 2)
